# 1 - Feeder/exit cams data (template)

Combine per-video detections into daily files and compute average counts. Update config paths before running.

In [ ]:
from pathlib import Path
import glob
import pandas as pd

import bb_metrics
cfg = bb_metrics.load_config('/path/to/your/season_config.py')  # replace with your config
from bb_metrics import feedercams

bb_metrics.set_config(cfg)


In [ ]:
# Inputs/outputs from config
inputdir = cfg.feedercam_input_dir
daily_dir = cfg.feedercam_daily_dir
avg_dir = cfg.feedercam_avg_dir
alldirs = sorted([d for d in Path(inputdir).glob('2025*') if d.is_dir()])
alldirs[:3]


In [ ]:
# Combine per-video files into per-day parquet
for whichpi in ['feedercam', 'exitcam']:
    for datedir in alldirs:
        status = feedercams.process_datedir(
            datedir,
            whichpi,
            outputdir=daily_dir,
            recalc=False,
            clahepostfix='-nc',  # '-c' or '-nc'
        )
        if status not in ('skip_existing', 'ok'):
            print(datedir.name, whichpi, status)


In [ ]:
# Compute average counts for daily files
for whichcam in ['feedercam', 'exitcam']:
    for clahepostfix in ['-c', '-nc']:
        pattern = f"2025*{whichcam}{clahepostfix}.parquet"
        daily_files = sorted(Path(daily_dir).glob(pattern))
        results = feedercams.process_daily_files(
            daily_files,
            avg_dir=avg_dir,
            recalc=True,
            clahepostfix=clahepostfix,
            localizer_threshold=0.1,
            bee_id_confidence_threshold=0.01,
            processes=2,
        )
        counts = pd.Series([r[0] for r in results]).value_counts()
        print(whichcam, clahepostfix, counts.to_dict())
